In [ ]:
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "index.scan.map.json").exists():
            return candidate
    raise RuntimeError("Could not locate repo root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())

In [ ]:
from __future__ import annotations

import random
 
from drive_service.map_index_service import MapIndex


 
SOURCE_INDEX_PATH = REPO_ROOT / "index.scan.map.json"
SAMPLE_INDEX_PATH = REPO_ROOT / "scan" / "samples.index.scan.map.json"
SAMPLE_SIZE = 20
SAMPLE_SEED = 42

source = MapIndex.load_index(str(SOURCE_INDEX_PATH), strict=True)

eligible_ids = [
    file_id
    for file_id, entry in source.files.items()
    if file_id and (entry.type != "folder")
]
if not eligible_ids:
    raise RuntimeError("No eligible files found in source index")

sample_count = min(SAMPLE_SIZE, len(eligible_ids))
rng = random.Random(SAMPLE_SEED)
selected_ids = rng.sample(eligible_ids, k=sample_count)

sample_files = {file_id: source.files[file_id].model_dump() for file_id in selected_ids}
sample_index = MapIndex.generate_index(source.root_id, source.employee_count, sample_files)
SAMPLE_INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
sample_index.save_index(str(SAMPLE_INDEX_PATH))

print(f"Source index: {SOURCE_INDEX_PATH}")
print(f"Generated sample index: {SAMPLE_INDEX_PATH}")
print(f"Selected files: {sample_count} / {len(eligible_ids)}")
print(f"Seed: {SAMPLE_SEED}")

Source index: C:\Users\Rosario\Desktop\dev\Nursind\index.scan.map.json
Generated sample index: C:\Users\Rosario\Desktop\dev\Nursind\scan\samples.index.scan.map.json
Selected files: 20 / 20563
Seed: 42


In [3]:
from __future__ import annotations

from pathlib import Path

from drive_service.auth_service import load_creds
from drive_service.downloads import download_pdf_stream
from drive_service.drive_client import get_drive_service
from drive_service.map_index_service import MapIndex
from drive_service.names import safe_name


 
SAMPLE_INDEX_PATH = REPO_ROOT / "scan" / "samples.index.scan.map.json"
SAMPLES_OUT_DIR = REPO_ROOT / "samples" / "from_index"
SKIP_EXISTING = True

sample_index = MapIndex.load_index(str(SAMPLE_INDEX_PATH), strict=True)
creds = load_creds()
drive = get_drive_service(creds)

downloaded = 0
skipped = 0
failed: list[tuple[str, str]] = []

for file_id, entry in sample_index.files.items():
    if entry.type == "folder":
        skipped += 1
        continue

    employee = safe_name(entry.employee or "unknown")
    file_name = safe_name(entry.file_name or file_id or "unknown.pdf")
    if not file_name.lower().endswith(".pdf"):
        file_name = f"{file_name}.pdf"

    out_dir = SAMPLES_OUT_DIR / employee
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / file_name

    if SKIP_EXISTING and out_path.exists():
        skipped += 1
        continue

    stream = None
    try:
        stream = download_pdf_stream(drive, file_id)
        out_path.write_bytes(stream.read())
        downloaded += 1
    except Exception as exc:
        failed.append((file_id, f"{type(exc).__name__}: {exc}"))
    finally:
        if stream is not None:
            try:
                stream.close()
            except Exception:
                pass

print(f"Sample index: {SAMPLE_INDEX_PATH}")
print(f"Output folder: {SAMPLES_OUT_DIR}")
print(f"Downloaded: {downloaded}")
print(f"Skipped: {skipped}")
print(f"Failed: {len(failed)}")
if failed:
    print("First failures:")
    for file_id, reason in failed[:10]:
        print(f"- {file_id}: {reason}")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=1032431065744-1t0h5ge7b9630k1j25odlfpaqblsrqb5.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A54088%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=QXsHOQqimFUXGmKKmdhCNMDMj3KeFk&access_type=offline
Sample index: C:\Users\Rosario\Desktop\dev\Nursind\scan\samples.index.scan.map.json
Output folder: C:\Users\Rosario\Desktop\dev\Nursind\samples\from_index
Downloaded: 20
Skipped: 0
Failed: 0
